# 5차시 실습 노트북
## 상태 공간의 배제 — 분기 한정법 · 탐욕법
세종과학고등학교 정보과학 · 1학년

---

지난 시간까지 우리는 상태 공간을 **전부** 훑었다. 전체 탐색은 언제나 옳지만, 언제나 **끝나지는** 않는다.

오늘은 **보지 않아도 되는 곳을 보지 않는 법**을 배운다.

| | 버리는 가지 | 버리는 근거 | 최적해 |
|---|---|---|---|
| **분기 한정법** | 답이 없음이 **증명된** 가지 | 논리적 증명 | **보장** |
| **탐욕법** | 최선 하나를 뺀 **전부** | 직관 (논리적 비약) | 보장 못 함 |

**오늘의 순서**
1~2번에서 배제를 처음 경험하고, 3~5번에서 분기 한정법을, 6~7번에서 탐욕법을 다룬다.

> 💡 **오늘의 관전 포인트는 &lsquo;답&rsquo;이 아니라 &lsquo;횟수&rsquo;다.**
> 거의 모든 셀이 답과 함께 **연산 횟수**를 출력한다. 배제 전후로 답은 그대로인데 횟수만 달라지는 것을 눈으로 확인하자.

## 0. 준비 — 연산 횟수를 시간으로 바꿔 보기

앞으로 &ldquo;1초에 1,000만 개의 연산을 처리하는 컴퓨터&rdquo;를 기준으로 시간을 이야기한다.
앞으로 계속 쓸 것이므로 함수로 만들어 두자.

아래 셀은 먼저 그냥 실행하자.

**실행 결과**
```
         10,000 번  ->  1.0 ms
     10,000,000 번  ->  1.0 초
  1,000,000,000 번  ->  1.7 분
 11,111,111,110 번  ->  18.5 분
 10,000,000,000 번  ->  16.7 분
```

In [ ]:
def est_time(ops):
    """1초에 1,000만 번 연산하는 컴퓨터를 가정하고 걸리는 시간을 문자열로 돌려준다."""
    sec = ops / 10_000_000
    if sec < 1:
        return f'{sec*1000:.1f} ms'
    if sec < 60:
        return f'{sec:.1f} 초'
    if sec < 3600:
        return f'{sec/60:.1f} 분'
    if sec < 86400:
        return f'{sec/3600:.1f} 시간'
    if sec < 86400*365:
        return f'{sec/86400:.1f} 일'
    return f'{sec/(86400*365):,.0f} 년'


for ops in [10**4, 10**7, 10**9, 11_111_111_110, 10**10]:
    print(f'{ops:>15,} 번  ->  {est_time(ops)}')

## 1. 배제의 첫 경험 — 소수의 개수 세기

> 100 이하의 자연수 n이 주어질 때, 1부터 n까지 자연수에 포함된 **소수의 개수**를 출력하시오.

k가 소수인지 판단할 때, 2부터 k−1까지 전부 나눠 볼 필요가 있을까?

> **k = a × b 이면, a와 b 둘 중 하나는 항상 √k 보다 작거나 같다.**
> (둘 다 √k보다 크면 a×b > √k×√k = k 가 되어 모순이다)

따라서 **2부터 √k까지** 나누어지지 않으면, 그 위는 **보지 않아도** k가 소수임이 확정된다.

**실행 결과** (빈칸을 고친 뒤)
```
1 ~ 100 의 소수 개수 : 25
나눗셈 횟수 : 525 회
```

> **지금 상태로 먼저 한 번 실행하세요.** 오류 없이 실행되고, 소수 개수도 **25로 맞게** 나옵니다.
> 다만 나눗셈 횟수가 **4851회**입니다.
>
> ①의 `i` 를 고쳐서 **525회**로 줄이세요. `int(i**(1/2))+1` 을 사용합니다.
>
> ⚠️ `+1` 을 빠뜨리면 어떻게 될까요? 고친 뒤 `+1` 을 빼고도 실행해서 **답이 틀어지는지** 확인해보세요.

In [ ]:
n = 100
ans = 0
ops = 0                     # 나눗셈을 몇 번 했는지 센다

for i in range(2, n+1):
    cnt = 0
    for j in range(2, i):              # ← ① 이 범위를 고쳐 보자
        ops = ops + 1
        if (i % j) == 0:
            cnt = 1
    if cnt == 0:
        ans = ans + 1

print('1 ~', n, '의 소수 개수 :', ans)
print('나눗셈 횟수 :', ops, '회')

### 1-2. n을 키우면 격차는 얼마나 벌어지는가

아래 셀은 그대로 실행하자. 두 방법의 나눗셈 횟수를 n을 키워 가며 비교한다.

**실행 결과**
```
       n |      2~k-1 |      2~√k |     배율
----------------------------------------------
     100 |      4,851 |       525 |   9.2 배
   1,000 |    498,501 |    19,615 |  25.4 배
  10,000 | 49,985,001 |   651,750 |  76.7 배
```
시간 복잡도가 **O(n²) → O(n√n)** 으로 바뀌었기 때문에, n이 커질수록 격차도 함께 커진다.

In [ ]:
print(f'{"n":>8} | {"2~k-1":>10} | {"2~√k":>9} | {"배율":>6}')
print('-' * 46)

for n in [100, 1000, 10000]:
    a = 0
    b = 0
    for k in range(2, n+1):
        a = a + (k - 2)                     # 2 ~ k-1 은 k-2 번
        b = b + (int(k**(1/2)) - 1)         # 2 ~ √k 는 √k-1 번
    print(f'{n:>8,} | {a:>10,} | {b:>9,} | {a/b:>5.1f} 배')

## 2. 약수의 합 — 절반은 공짜로 얻는다

> 100억 이하의 자연수 n이 주어질 때, **n의 모든 약수의 합**을 구하시오.

1부터 n까지 전부 확인하면 100억 번. 1초에 1,000만 번 연산하는 컴퓨터로 **약 16.7분**이다.

> **n = a × b 일 때, a를 찾으면 b도 같이 찾은 것이다.**
> 10 = 1×10, 10 = 2×5 → √10 ≈ 3.16 이므로 **1, 2만 찾으면** 10, 5는 계산 없이 따라온다.

단, **√n이 자연수일 때**(n이 제곱수일 때) √n을 두 번 더하게 되므로 보정이 필요하다.

**실행 결과** (빈칸을 채운 뒤)
```
n = 1000
전체 탐색 : 2340 / 연산 1,000 회
배제 후   : 2340 / 연산 31 회
```

> **빈칸 2곳입니다.**
> - ②-1 : 짝이 되는 약수 `n // i` 도 함께 더한다 (1줄)
> - ②-2 : n이 제곱수이면 √n 이 두 번 더해졌으므로 한 번 빼 준다 (2줄)
>
> `if n == int(n**(1/2))**2:` 로 제곱수를 판별할 수 있습니다.
>
> 채운 뒤 **n = 36** 으로 바꿔 실행하세요. 36은 제곱수(6×6)라 ②-2가 없으면 **답이 6만큼 커집니다.**

In [ ]:
n = 1000

# ① 전체 탐색 — 1부터 n까지 전부 확인한다
ans1 = 0
ops1 = 0
for i in range(1, n+1):
    ops1 = ops1 + 1
    if n % i == 0:
        ans1 = ans1 + i

# ② 상태 공간 배제 — 1부터 √n 까지만 확인한다
ans2 = 0
ops2 = 0
for i in range(1, int(n**(1/2))+1):
    ops2 = ops2 + 1
    if n % i == 0:
        ans2 = ans2 + i
        # ②-1 짝이 되는 약수도 함께 더한다 (1줄)


# ②-2 n이 제곱수라면 √n 이 두 번 더해졌다 → 한 번 빼 준다 (2줄)


print('n =', n)
print('전체 탐색 :', ans1, '/ 연산', f'{ops1:,}', '회')
print('배제 후   :', ans2, '/ 연산', f'{ops2:,}', '회')

### 2-2. 100억으로 확인하기

전체 탐색은 100억 번이라 **돌릴 수 없다.** 배제한 쪽만 실제로 돌려 보고, 전체 탐색은 **횟수만** 계산해서 비교한다.

**실행 결과**
```
n = 10,000,000,000
약수의 합 = 24,987,792,457

전체 탐색 O(n)  : 10,000,000,000 회  ->  16.7 분
배제    O(√n)  :        100,000 회  ->  10.0 ms
                                      약 100,000 배
```

In [ ]:
n = 10**10

ans = 0
for i in range(1, int(n**(1/2))+1):
    if n % i == 0:
        ans = ans + i + (n // i)
if n == int(n**(1/2))**2:
    ans = ans - int(n**(1/2))

full = n                      # 전체 탐색이라면 해야 할 연산 횟수
fast = int(n**(1/2))          # 배제했을 때의 연산 횟수

print(f'n = {n:,}')
print(f'약수의 합 = {ans:,}')
print()
print(f'전체 탐색 O(n)  : {full:>15,} 회  ->  {est_time(full)}')
print(f'배제    O(√n)  : {fast:>15,} 회  ->  {est_time(fast)}')
print(f'{"약 " + format(full//fast, ",") + " 배":>52}')

## 3. 【분기 한정법】 자연수의 합으로 n 만들기

> 10 이하의 자연수 n이 주어질 때, **자연수의 합으로 n을 나타낼 수 있는 모든 경우의 개수**를 구하시오.
> (더하는 순서가 다르면 다른 경우로 센다. 예: 1+2 와 2+1 은 서로 다른 경우)

n=10이면 해가 될 수 있는 경우는 10 + 10² + … + 10¹⁰ = **11,111,111,110가지**. 약 **20분**이 걸린다.

### 배제의 근거

> 합이 이미 n을 **초과**했다면, 앞으로 **어떤 자연수를 더해도** 자연수는 양수이므로 합은 절대 n으로 돌아올 수 없다.

그러므로 이 아래를 탐색하는 것은 **확실히** 낭비다. 탐색을 멈추고 **백트랙**한다.
이렇게 &ldquo;더 볼 필요가 없다고 판단되는 경로&rdquo;를 잘라내는 것이 **분기 한정법**이다.

**실행 결과** (빈칸을 채운 뒤, `n = 6`)
```
n = 6
경우의 수 : 32
함수 호출 : 1,158 회
```

> **지금은 `n = 6` 입니다.**
>
> **빈칸 2곳**
> - ① 합이 n을 넘었으면 0을 반환한다 (2줄) → **분기 한정**
> - ② `i` 를 더해 다음 상태로 내려간다 (1줄). 지금은 `cnt + 0` 이라 항상 0이 나옵니다
>
> **순서가 중요합니다.** ②를 먼저 채우고 실행하면 답이 **32로 맞게** 나오지만 함수 호출이 **67,182회**입니다.
> 그다음 ①을 채우고 다시 실행해서, 답은 그대로인데 **호출 횟수만 1,158회로 줄어드는 것**을 확인하세요.
>
> 둘 다 채웠다면 `n = 10` 으로 바꿔 실행하세요. **512 / 51,210회**가 나오면 성공입니다.
> (①을 채우지 않고 `n = 10` 을 돌리면 **20분**이 걸립니다. 절대 실행하지 마세요.)

In [ ]:
n = 6
calls = 0

def f(s, k):
    global calls
    calls = calls + 1

    if k == 0:
        if n == s:
            return 1
        else:
            return 0

    # ① 합이 n을 넘었다면? 남은 k개를 어떻게 골라도 n이 될 수 없다 (2줄)


    cnt = 0
    for i in range(1, n+1):
        cnt = cnt + 0                # ← ② 여기를 고치자
    return cnt

ans = 0
for i in range(1, n+1):
    ans = ans + f(0, i)

print('n =', n)
print('경우의 수 :', ans)
print('함수 호출 :', f'{calls:,}', '회')

### 3-2. 한 줄을 지우면 얼마나 느려지는가

`if s > n: return 0` **한 줄**의 값어치를 직접 재어 보자. 아래 셀은 그대로 실행한다.

**실행 결과**
```
  n |     배제 없음 |     분기 한정 |     배율 | 경우의 수
------------------------------------------------------------
  4 |         452 |         132 |    3.4 배 |     8
  5 |       4,880 |         405 |   12.0 배 |    16
  6 |      67,182 |       1,158 |   58.0 배 |    32
  7 |   1,120,931 |       3,143 |  356.6 배 |    64
------------------------------------------------------------
 10 | 11,111,111,110 회 (18.5 분)  ->  51,210 회 (5.1 ms)
```
**답은 언제나 같다.** 달라지는 것은 거기에 도달하기까지의 횟수뿐이다.
그리고 그 격차는 n이 커질수록 **폭발적으로** 벌어진다.

In [ ]:
import sys
sys.setrecursionlimit(100000)

def count_calls(n, prune):
    calls = [0]
    def f(s, k):
        calls[0] = calls[0] + 1
        if k == 0:
            return 1 if s == n else 0
        if prune and s > n:
            return 0
        cnt = 0
        for i in range(1, n+1):
            cnt = cnt + f(s+i, k-1)
        return cnt
    total = 0
    for i in range(1, n+1):
        total = total + f(0, i)
    return total, calls[0]

print(f'{"n":>3} | {"배제 없음":>12} | {"분기 한정":>11} | {"배율":>7} | 경우의 수')
print('-' * 60)
for n in [4, 5, 6, 7]:
    a_ans, a = count_calls(n, False)
    b_ans, b = count_calls(n, True)
    print(f'{n:>3} | {a:>12,} | {b:>11,} | {a/b:>6.1f} 배 | {a_ans:>5}')

print('-' * 60)
full10 = sum(10**k for k in range(1, 11))
_, fast10 = count_calls(10, True)
print(f' 10 | {full10:,} 회 ({est_time(full10)})  ->  {fast10:,} 회 ({est_time(fast10)})')

## 4. 연습 문제 — 주어진 수들로 s 만들기

> 10 이하의 **서로 다른 n개**의 자연수가 주어질 때, 자연수들의 합으로 **s**를 만들 수 있는
> 모든 경우의 수를 구하는 프로그램을 작성하시오.

**입력**: `n = 5`, `s = 10`, `A = [1, 2, 3, 4, 5]`

3번과 뼈대가 완전히 같다. 다른 것은 **더할 수를 `1~n`에서 고르느냐, 리스트 `A`에서 고르느냐**뿐이다.

**실행 결과**
```
경우의 수 : 87
함수 호출 : 800 회
```

> **빈칸 2곳입니다.**
> - ① 수를 k개 다 골랐다. **성공인지 판단**하려면 무엇과 무엇을 비교해야 하는가?
> - ② `A[i]` 를 합에 더하고, 앞으로 고를 개수를 하나 줄여 내려간다
>
> **왜 87인가?** A에서 중복을 허용하고 순서를 구분해 합이 10이 되는 경우는
> 2개짜리 1가지 + 3개짜리 18가지 + 4개짜리 68가지 = **87**입니다.
>
> 채운 뒤 `if c > s: return 0` 을 **주석 처리**하고 실행해 보세요. 답은 같고 호출 횟수만 늘어납니다.

In [ ]:
n = 5
s = 10
A = [1, 2, 3, 4, 5]
calls = 0

def g(c, k):
    global calls
    calls = calls + 1

    if k == 0:
        # ① 수를 k개 다 골랐다. 성공인지 판단하려면?
        if False:                    # ← 여기를 고치자
            return 1
        else:
            return 0

    if c > s:                        # 더 이상 탐색을 진행하지 않고 백트랙
        return 0

    cnt = 0
    for i in range(0, n):
        cnt = cnt + 0                # ← ② 여기를 고치자
    return cnt

ans = 0
for i in range(0, n):
    ans = ans + g(0, i)

print('경우의 수 :', ans)
print('함수 호출 :', f'{calls:,}', '회')

## 5. 같은 문제, 세 가지 전략 — 최소 동전의 개수

> 사용할 수 있는 동전은 **10원, 50원, 100원, 500원**이다.
> **n원을 지불할 때 최소 동전의 개수**를 구하시오.

같은 문제를 ① 전체 탐색 ② 분기 한정법 ③ 탐욕법 세 가지로 풀고, **연산 횟수를 비교**한다.

### 5-1. 전체 탐색

동전 하나를 고르고, 또 하나를 고르는 과정을 반복한다. n원이 되면 개수를 비교하고,
**n원을 초과하면 백트랙**한다. 아래 셀은 그대로 실행한다.

**실행 결과**
```
n = 370
최소 동전 개수 : 6 / 함수 호출 402,241 회

n = 1780 이라면 함수 호출은?
  738,339,139,800,419,017,181,002 회  ->  2,341,258,054 년
```
n=1780에서는 **돌릴 수 없다.** 그래서 횟수만 수식으로 계산해 본다.

In [ ]:
import sys
sys.setrecursionlimit(100000)

coin = [500, 100, 50, 10]

def full_search(n):
    best = [987654321]
    calls = [0]
    def f(won, d):
        calls[0] = calls[0] + 1
        if won > n:
            return
        if won == n:
            best[0] = min(d, best[0])
            return
        for i in range(4):
            f(won + coin[i], d + 1)
    f(0, 0)
    return best[0], calls[0]


ans, calls = full_search(370)
print('n = 370')
print('최소 동전 개수 :', ans, '/ 함수 호출', f'{calls:,}', '회')


# n=1780 은 실제로 돌릴 수 없으므로, 노드 수만 세어 본다
def full_search_nodes(n):
    c = [0] * (n + 1)
    c[0] = 1
    for w in range(1, n+1):
        for x in coin:
            if w - x >= 0:
                c[w] = c[w] + c[w - x]
    return sum(c)

big = full_search_nodes(1780)
print()
print('n = 1780 이라면 함수 호출은?')
print(f'  {big:,} 회  ->  {est_time(big)}')

### 5-2. 분기 한정법

> **k개의 동전으로 n원을 만드는 방법을 이미 찾았다면**, k개 이상을 쓰는 방법은 더 볼 필요가 없다.

지금까지 찾은 최선의 답 `ans` 가 곧 **한계선(bound)** 이 된다.
좋은 답을 빨리 찾을수록 더 많이 잘린다.

**실행 결과** (빈칸을 채운 뒤)
```
n = 370
최소 동전 개수 : 6 / 함수 호출 1,369 회
```

> **지금은 `n = 370` 입니다.** 빈칸을 채우기 전에 먼저 실행하면 **402,241회** — 5-1과 똑같습니다.
> ①을 채우면 **1,369회**로 줄어듭니다. (약 294배)
>
> - ① 이미 찾은 최선(`ans`)보다 동전을 많이 썼거나 같다면 더 볼 필요가 없다 (2줄)
>
> 채운 뒤 `n = 1780` 으로 바꿔 실행하세요. **9개 / 290,697회**가 나옵니다.
> (①이 없으면 7.4×10²³회입니다. 절대 실행하지 마세요.)

In [ ]:
coin = [500, 100, 50, 10]

n = 370
ans = 987654321
calls = 0

def f(won, d):
    global ans, calls
    calls = calls + 1

    if won > n:
        return

    # ① 이미 찾은 최선(ans)보다 동전을 많이 썼다면? (2줄)


    if won == n:
        ans = min(d, ans)
        return

    for i in range(4):
        f(won + coin[i], d + 1)
    return

f(0, 0)
print('n =', n)
print('최소 동전 개수 :', ans, '/ 함수 호출', f'{calls:,}', '회')

### 5-3. 탐욕법

> **가장 가치가 높은 동전부터** 내림차순으로, n원을 초과하지 않도록 최대한 많이 내준다.

재귀도, 백트랙도 없다. 반복문 **4번**이면 끝난다.
시간 복잡도가 O(2ⁿ)에서 **O(m)** (m은 동전의 가짓수)으로 줄어든다.

**실행 결과** (빈칸을 채운 뒤)
```
n = 1780
최소 동전 개수 : 9 / 연산 4 회
500원 x 3 + 100원 x 2 + 50원 x 1 + 10원 x 3
```

> **빈칸 2곳, 각 1줄입니다.**
> - ① 이 동전을 최대 몇 개 쓸 수 있는가 → `n // coin[i]`
> - ② 그러고 나면 남는 금액은 → `n % coin[i]`
>
> ②를 빠뜨리면 어떻게 될까요? 채우기 전에 한 번 실행해 보고, 왜 그런 값이 나오는지 설명해 보세요.

In [ ]:
coin = [500, 100, 50, 10]

n = 1780
money = n
ans = 0
ops = 0
detail = []

for i in range(4):
    ops = ops + 1
    # ① 이 동전을 최대 몇 개 쓸 수 있는가 (1줄)


    # ② 그러고 나면 남는 금액은 (1줄)


    detail.append(f'{coin[i]}원 x {0}')

print('n =', money)
print('최소 동전 개수 :', ans, '/ 연산', ops, '회')
print(' + '.join(detail))

### 5-4. 탐욕법은 언제 틀리는가

우리나라 동전으로 탐욕법이 통한 것은 **우연이 아니다.**
10 · 50 · 100 · 500은 **큰 단위가 작은 단위의 배수**라서, 큰 동전을 미루는 것이 결코 이득이 아님을 증명할 수 있다.

그렇다면 동전이 **[1, 4, 5]** 라면? 아래 셀을 그대로 실행해 보자.

**실행 결과**
```
동전 [5, 4, 1] 로 만들 때 탐욕법이 틀리는 금액
     금액 |   탐욕법 |   최적해
------------------------------
      8 |     4 |     2
     12 |     4 |     3
     13 |     5 |     3
     17 |     5 |     4
     18 |     6 |     4

8원 : 탐욕법 5+1+1+1 = 4 개 / 최적해 4+4 = 2 개
```
> **탐욕법을 이용하여 최적해를 구하기 위해서는, 다음 상태를 선택할 때 논리적 근거가 필요하다.**

In [ ]:
coins = [5, 4, 1]

def greedy(n):
    a = 0
    for x in coins:
        a = a + n // x
        n = n % x
    return a

def optimal(n):
    INF = 10**9
    dp = [0] + [INF] * n
    for w in range(1, n+1):
        for x in coins:
            if w - x >= 0:
                dp[w] = min(dp[w], dp[w-x] + 1)
    return dp[n]

print('동전', coins, '로 만들 때 탐욕법이 틀리는 금액')
print(f'{"금액":>7} | {"탐욕법":>5} | {"최적해":>5}')
print('-' * 30)
for n in range(1, 21):
    g, o = greedy(n), optimal(n)
    if g != o:
        print(f'{n:>7} | {g:>5} | {o:>5}')

print()
print('8원 : 탐욕법 5+1+1+1 =', greedy(8), '개 / 최적해 4+4 =', optimal(8), '개')

## 6. 【탐욕법】 트리의 최대 경로 합

> 루트 노드부터 단말 노드로 이동할 때, **이동 경로에 있는 노드 합의 최댓값**을 구하시오.

```
              3
          ┌───┴───┐
          8       2
        ┌─┴─┐   ┌─┴─┐
        4   3   3   9        ← 가운데 3은 8과 2가 공유하는 자식
      ┌─┴─┐ ┌─┴─┐ ┌─┴─┐
      1   5 5  2 2   8
```
같은 삼각형을 리스트로 적으면 이렇다.

```python
tri = [[3],
       [8, 2],
       [4, 3, 9],
       [1, 5, 2, 8]]
```
`tri[r][c]` 의 자식은 `tri[r+1][c]` (왼쪽) 과 `tri[r+1][c+1]` (오른쪽) 이다.

**실행 결과** (빈칸을 채운 뒤)
```
탐욕법   : 20 [3, 8, 4, 5] / 비교 3 회
전체 탐색 : 22 [3, 2, 9, 8] / 호출 15 회
-> 탐욕법은 최적해를 찾지 못했다!
```

> **빈칸 1곳입니다.**
> - ① 두 자식 `left` 와 `right` 중 **큰 쪽**을 고르도록 `c` 를 갱신한다
>   (왼쪽을 고르면 `c` 는 그대로, 오른쪽을 고르면 `c + 1`)
>
> 채우기 전에 실행하면 항상 왼쪽만 골라 `[3, 8, 4, 1] = 16` 이 나옵니다.
>
> 채운 뒤 **탐욕법의 답과 전체 탐색의 답을 비교**하세요. 왜 달라졌습니까?
> 어느 갈림길에서 탐욕법이 길을 잃었는지 말로 설명해 보세요.

In [ ]:
tri = [[3],
       [8, 2],
       [4, 3, 9],
       [1, 5, 2, 8]]

# ---------- 탐욕법 ----------
r, c = 0, 0
total = tri[0][0]
path = [tri[0][0]]
compares = 0

while r < len(tri) - 1:
    left = tri[r+1][c]
    right = tri[r+1][c+1]
    compares = compares + 1

    # ① 두 자식 중 큰 쪽을 고른다 → c 를 갱신 (2~4줄)


    r = r + 1
    total = total + tri[r][c]
    path.append(tri[r][c])

print('탐욕법   :', total, path, '/ 비교', compares, '회')


# ---------- 전체 탐색 (비교용 — 그대로 실행) ----------
best = [0, None]
calls = [0]

def all_paths(r, c, s, p):
    calls[0] = calls[0] + 1
    if r == len(tri) - 1:
        if s > best[0]:
            best[0] = s
            best[1] = list(p)
        return
    for nc in (c, c+1):
        p.append(tri[r+1][nc])
        all_paths(r+1, nc, s + tri[r+1][nc], p)
        p.pop()

all_paths(0, 0, tri[0][0], [tri[0][0]])
print('전체 탐색 :', best[0], best[1], '/ 호출', calls[0], '회')

if total != best[0]:
    print('-> 탐욕법은 최적해를 찾지 못했다!')

### 6-2. 다른 트리로 한 번 더 확인하기

이번 트리는 **완전 이진 트리**다. `tri[r][i]` 의 자식은 `tri[r+1][2i]`, `tri[r+1][2i+1]`.

```
                  1
          ┌───────┴───────┐
          2               1
      ┌───┴───┐       ┌───┴───┐
      2       5       3       5
    ┌─┴─┐   ┌─┴─┐   ┌─┴─┐   ┌─┴─┐
    5   4   2   1   8   3   2   5
```
아래 셀을 그대로 실행해 답을 확인하자.

**실행 결과**
```
탐욕법   : 10 [1, 2, 5, 2]
전체 탐색 : 13 [1, 1, 3, 8]

경로 8 가지
  1-2-2-5 = 10
  1-2-2-4 =  9
  1-2-5-2 = 10  <- 탐욕법
  1-2-5-1 =  9
  1-1-3-8 = 13  <- 최댓값
  1-1-3-3 =  8
  1-1-5-2 =  9
  1-1-5-5 = 12
```
첫 갈림길에서 **2 > 1** 이라는 이유로 왼쪽을 골랐지만, 오른쪽에는 **3과 8**이 기다리고 있었다.

In [ ]:
lv = [[1],
      [2, 1],
      [2, 5, 3, 5],
      [5, 4, 2, 1, 8, 3, 2, 5]]

# ---------- 탐욕법 ----------
i = 0
total = lv[0][0]
path = [lv[0][0]]
for r in range(1, len(lv)):
    left = lv[r][i*2]
    right = lv[r][i*2 + 1]
    i = i*2 if left >= right else i*2 + 1
    total = total + lv[r][i]
    path.append(lv[r][i])
print('탐욕법   :', total, path)

# ---------- 전체 탐색 ----------
routes = []
def walk(r, i, s, p):
    if r == len(lv) - 1:
        routes.append(('-'.join(str(x) for x in p), s))
        return
    for ni in (i*2, i*2 + 1):
        p.append(lv[r+1][ni])
        walk(r+1, ni, s + lv[r+1][ni], p)
        p.pop()
walk(0, 0, lv[0][0], [lv[0][0]])

best = max(routes, key=lambda t: t[1])
print('전체 탐색 :', best[1], [int(x) for x in best[0].split('-')])
print()
print('경로', len(routes), '가지')
for name, s in routes:
    mark = '  <- 최댓값' if s == best[1] else ('  <- 탐욕법' if name == '-'.join(str(x) for x in path) else '')
    print(f'  {name} = {s:>2}{mark}')

## 7. 개구리의 최대 이동 횟수

> 개구리 세 마리가 순차적으로 배치된 연잎 위에서 놀고 있다. 개구리는 현재 **서로 다른 연잎** 위에 있다.
> **바깥쪽에 있는 개구리 두 마리 중 한 마리**가 **다른 두 개구리 사이의 연잎**으로 건너뛰어 이동할 수 있다.
> 개구리는 같은 연잎 위에 두 마리 이상 있을 수 없다.
> 개구리가 **이동할 수 있는 최대 횟수**는 얼마일까?

**입력**: `1 8 10` → **실행 결과**: `6`

### 규칙을 먼저 찾자

위치를 정렬해 `a < b < c` 라 하고, 두 간격을 `L = b - a`, `R = c - b` 라 하자.

바깥 개구리가 사이로 뛰면, **두 간격 중 하나만 남고 그 값이 1 줄어든다.**
- 왼쪽 개구리가 뛰면 → 남는 간격은 `R - 1`
- 오른쪽 개구리가 뛰면 → 남는 간격은 `L - 1`

이동을 **최대한 많이** 하려면 매번 **큰 쪽을 남겨야** 한다. 그리고 간격이 1이 되면 더 뛸 수 없다.

> **최대 이동 횟수 = max(L, R) − 1**

매번 &lsquo;조금씩만 줄이는&rsquo; 선택이 곧 **탐욕적 선택**이며, 이 문제에서는 그 선택이 최적임을 증명할 수 있다.

> **빈칸 2곳입니다.**
> - ① 위치를 순서대로 정렬한다 (1줄) — 입력이 `8 1 10` 처럼 들어올 수도 있다
> - ② 최대 이동 횟수를 계산한다 (`if / else` 로 써도, `max()` 한 줄로 써도 좋다)
>
> 채운 뒤 아래 검증 셀을 실행해, 실제로 개구리를 움직여 본 결과와 일치하는지 확인하세요.

In [ ]:
data = [1, 8, 10]

# ① 위치 순서대로 정렬한다 (1줄)


left = data[1] - data[0]
right = data[2] - data[1]

# ② 최대 이동 횟수는? (1~4줄)
ans = 0


print('개구리 위치 :', data)
print('간격 :', left, ',', right)
print('최대 이동 횟수 :', ans)

### 7-2. 규칙이 맞는지 직접 움직여 검증하기

규칙은 &lsquo;그럴듯해&rsquo; 보인다고 믿는 것이 아니라 **확인하는** 것이다.
실제로 개구리를 한 마리씩 옮겨 보며 몇 번 움직일 수 있는지 세어 보자. 아래 셀은 그대로 실행한다.

**실행 결과**
```
시작 : [1, 8, 10]
 1회 : [1, 2, 8]   (오른쪽 개구리 10 -> 2)
 2회 : [2, 7, 8]   (왼쪽 개구리 1 -> 7)
 3회 : [2, 3, 7]   (오른쪽 개구리 8 -> 3)
 4회 : [3, 6, 7]   (왼쪽 개구리 2 -> 6)
 5회 : [3, 4, 6]   (오른쪽 개구리 7 -> 4)
 6회 : [4, 5, 6]   (왼쪽 개구리 3 -> 5)
더 이상 움직일 수 없다. 총 6 회

공식 max(L, R) - 1 = 6  ->  일치
```

In [ ]:
def simulate(a, b, c, verbose=True):
    pos = sorted([a, b, c])
    moves = 0
    if verbose:
        print('시작 :', pos)

    while True:
        L = pos[1] - pos[0]
        R = pos[2] - pos[1]
        if L <= 1 and R <= 1:
            break

        if L >= R:
            # 큰 쪽(L)을 남긴다 → 오른쪽 개구리가 왼쪽 개구리 바로 옆으로
            frm, to = pos[2], pos[0] + 1
            pos = sorted([pos[0], to, pos[1]])
            who = '오른쪽'
        else:
            # 큰 쪽(R)을 남긴다 → 왼쪽 개구리가 오른쪽 개구리 바로 옆으로
            frm, to = pos[0], pos[2] - 1
            pos = sorted([pos[1], to, pos[2]])
            who = '왼쪽'

        moves = moves + 1
        if verbose:
            print(f'{moves:>2}회 : {pos}   ({who} 개구리 {frm} -> {to})')

    if verbose:
        print('더 이상 움직일 수 없다. 총', moves, '회')
    return moves


m = simulate(1, 8, 10)
d = sorted([1, 8, 10])
formula = max(d[1]-d[0], d[2]-d[1]) - 1
print()
print('공식 max(L, R) - 1 =', formula, ' -> ', '일치' if m == formula else '불일치!')

print()
print('다른 입력으로도 확인')
for a, b, c in [(8, 1, 10), (1, 2, 3), (1, 5, 20), (3, 4, 9)]:
    m = simulate(a, b, c, verbose=False)
    d = sorted([a, b, c])
    f = max(d[1]-d[0], d[2]-d[1]) - 1
    print(f'  {sorted([a,b,c])} -> 시뮬레이션 {m} 회 / 공식 {f} 회  {"일치" if m==f else "불일치!"}')

---
수고했습니다!

**오늘의 한 문장**
> **배제는 답을 바꾸지 않는다. 시간을 바꾼다.** 단, 그 배제에 **논리적 근거**가 있을 때만.

오늘 만든 코드에서 배제는 늘 **단 한 줄**이었다.

| | 배제하는 한 줄 | 효과 |
|---|---|---|
| 소수 개수 | `range(2, int(i**0.5)+1)` | 4,851회 → 525회 |
| 약수의 합 | `range(1, int(n**0.5)+1)` + 짝 더하기 | 100억회 → 10만회 |
| 합 분해 | `if s > n: return 0` | 111억회 → 51,210회 |
| 최소 동전 | `if d >= ans: return` | 7.4×10²³회 → 290,697회 |
| 트리 경로 | `큰 쪽만 남긴다` | 2ⁿ → n **(단, 최적해를 잃는다)** |

**세 전략 정리**

| | 버리는 가지 | 근거 | 최적해 | 속도 |
|---|---|---|---|---|
| 전체 탐색 | 없음 | — | 보장 | 가장 느림 |
| **분기 한정법** | 답이 없음이 증명된 가지 | 논리적 증명 | **보장** | 문제에 따라 극적 |
| **탐욕법** | 최선 하나를 뺀 전부 | 직관 (비약) | **보장 못 함** | 가장 빠름 |

---

**다음 시간에는 관계기반 알고리즘을 배웁니다.**

6번 문제에서 탐욕법은 20을, 전체 탐색은 22를 답했다.
**22를 O(n²)에 찾는 방법**이 남아 있다. 큰 문제를 작은 문제로 나누고, 그 사이의 **관계**를 이용하는 것이다.

5번의 최소 동전 문제도 마찬가지다. 분기 한정법으로 29만 회가 걸린 문제를 **1,780번의 계산**으로 푸는 방법이 있다.